# 03 — Model Training

Builds and trains a CNN with PyTorch for binary classification (cat vs. dog).

> **Colab**: Set runtime to **T4 GPU** before running (`Runtime → Change runtime type → T4 GPU`).

## Environment Setup

In [ ]:
import os
import torch

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    if not os.path.exists('/content/cnn-classification-Eyad-M'):
        token = userdata.get('GITHUB_TOKEN')
        !git clone https://{token}@github.com/Eyad-M/cnn-classification-Eyad-M /content/cnn-classification-Eyad-M
    os.chdir('/content/cnn-classification-Eyad-M')
    drive.mount('/content/drive')
    %pip install -q -r requirements.txt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

DATA_DIR = '/content/drive/MyDrive/datasets/catsvsdogs' if IN_COLAB else '../data/raw'

## Data Loaders

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

class CatsDogsDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row['label'], dtype=torch.float32)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

splits_dir = Path(DATA_DIR) / 'splits'
train_loader = DataLoader(CatsDogsDataset(splits_dir / 'train.csv', train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(CatsDogsDataset(splits_dir / 'val.csv',   val_transform),   batch_size=BATCH_SIZE, num_workers=2)

## CNN Architecture

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

model = CNN().to(device)
print(model)

## Loss Function & Optimizer

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## Training Loop

In [ ]:
EPOCHS = 20
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0, 0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        preds = (torch.sigmoid(out) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            val_loss += criterion(out, labels).item()
            preds = (torch.sigmoid(out) >= 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    history['train_loss'].append(train_loss / len(train_loader))
    history['val_loss'].append(val_loss / len(val_loader))
    history['train_acc'].append(correct / total)
    history['val_acc'].append(val_correct / val_total)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {history['train_loss'][-1]:.4f} Acc: {history['train_acc'][-1]:.4f} | "
          f"Val Loss: {history['val_loss'][-1]:.4f} Acc: {history['val_acc'][-1]:.4f}")

## Training & Validation Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_title('Accuracy')
axes[1].legend()
plt.tight_layout()
plt.show()

## Save Model to `data/models/`

In [ ]:
models_dir = Path(DATA_DIR) / 'models'
models_dir.mkdir(exist_ok=True)
torch.save(model.state_dict(), models_dir / 'cnn.pth')
print(f'Model saved to {models_dir / "cnn.pth"}')